# 第二步：Bill 关联、政府背书证据与 Polarity 人工复核队列

本 notebook 在第一步输出的 Commons 四党审计数据上继续处理。主要目标：

1. 将 party-level 数据折叠为每个 division_key 一行；
2. 通过标准化标题将 legislation_name 与 bill_info 的 shortTitle 关联；
3. 从 Bill sponsor 和 government organisation 提取政府背书证据；
4. 排除不适合政策立场建模的程序性 motion；
5. 为 unknown 和 medium-confidence polarity 生成去重的人工复核队列；
6. 为 high-confidence polarity 生成质量抽样文件。

注意：Bill sponsor 说明的是基础 Bill 的来源，不一定说明当前 amendment 的来源。所有自动判断都保留规则、证据和置信度。

## 1. 导入依赖并设置路径

In [13]:
from pathlib import Path
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

import ast
import hashlib
import html
import json
import re
import unicodedata
from difflib import SequenceMatcher

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 140)

AUDIT_CANDIDATES = [
    Path('processed/commons_party_motion_audit.csv'),
    Path('reorganised/processed/commons_party_motion_audit.csv'),
]
BILL_INFO_CANDIDATES = [
    Path('data/raw/bill_info.csv'),
    Path('../scrap/3.6/bill_info.csv'),
    Path('scrap/3.6/bill_info.csv'),
]
AUDIT_PATH = next((path.resolve() for path in AUDIT_CANDIDATES if path.exists()), None)
BILL_INFO_PATH = next((path.resolve() for path in BILL_INFO_CANDIDATES if path.exists()), None)
if AUDIT_PATH is None:
    raise FileNotFoundError('找不到 commons_party_motion_audit.csv，请先运行第一步 notebook。')
if BILL_INFO_PATH is None:
    raise FileNotFoundError('找不到 bill_info.csv。')

OUTPUT_DIR = AUDIT_PATH.parent / 'polarity_review'
RANDOM_SEED = 70202
HIGH_CONFIDENCE_QC_SIZE = 50

PROCEDURAL_EXCLUDE_TYPES = {
    'private_sitting', 'closure', 'adjournment', 'programme',
    'standing_order_change', 'reasons_committee', 'timetable_change',
}

print(f'第一步审计数据：{AUDIT_PATH}')
print(f'Bill 信息：{BILL_INFO_PATH}')
print(f'输出目录：{OUTPUT_DIR}')

第一步审计数据：/Users/oooo1/01Course/2_70202Applying/reorganised/processed/commons_party_motion_audit.csv
Bill 信息：/Users/oooo1/01Course/2_70202Applying/scrap/3.6/bill_info.csv
输出目录：/Users/oooo1/01Course/2_70202Applying/reorganised/processed/polarity_review


## 2. 读取数据并检查必要字段

In [14]:
audit = pd.read_csv(AUDIT_PATH, low_memory=False)
bill_info = pd.read_csv(BILL_INFO_PATH, low_memory=False)
audit['motion_date'] = pd.to_datetime(audit['motion_date'], errors='coerce')

REQUIRED_AUDIT_COLUMNS = {
    'division_key', 'motion_date', 'motion_title_clean', 'motion_text_clean',
    'motion_type', 'legislation_name_clean', 'party', 'split',
    'motion_polarity', 'polarity_rule', 'polarity_confidence',
    'government_party', 'party_role',
}
REQUIRED_BILL_COLUMNS = {'billId', 'shortTitle', 'sponsors'}
missing_audit = sorted(REQUIRED_AUDIT_COLUMNS - set(audit.columns))
missing_bill = sorted(REQUIRED_BILL_COLUMNS - set(bill_info.columns))
if missing_audit:
    raise ValueError(f'第一步输出缺少字段：{missing_audit}')
if missing_bill:
    raise ValueError(f'bill_info 缺少字段：{missing_bill}')

print(f"审计数据：{audit.shape[0]:,} 行，{audit['division_key'].nunique():,} 个 division")
print(f'Bill 信息：{bill_info.shape[0]:,} 行')
print('政党：', sorted(audit['party'].dropna().unique()))
assert 'reform' not in set(audit['party'].dropna()), '审计数据仍包含 Reform，请重新运行第一步 notebook。'

审计数据：8,364 行，2,091 个 division
Bill 信息：129 行
政党： ['conservative', 'green', 'labour', 'liberal-democrat']


## 3. 折叠为每个 Division 一行

Polarity、motion text 和 Bill 关联都是 division-level 属性。人工标注时只需要标记一次，不需要对四个政党重复标记。

In [15]:
DIVISION_COLUMNS = [
    'division_key', 'motion_date', 'motion_year', 'motion_title_clean',
    'motion_text_clean', 'motion_type', 'division_name_clean',
    'legislation_name_clean', 'legislation_id', 'split',
    'motion_polarity', 'polarity_rule', 'polarity_confidence',
    'government_party', 'main_opposition_party',
]

consistency_columns = [
    'motion_date', 'motion_title_clean', 'motion_text_clean', 'motion_type',
    'legislation_name_clean', 'split', 'motion_polarity',
    'polarity_rule', 'polarity_confidence', 'government_party',
]
division_consistency = audit.groupby('division_key')[consistency_columns].nunique(dropna=False)
inconsistent_cells = int(division_consistency.gt(1).sum().sum())
if inconsistent_cells:
    raise ValueError(f'发现 {inconsistent_cells} 个 division-level 字段不一致，不能直接折叠。')

divisions = (
    audit.sort_values(['motion_date', 'division_key'])
    .drop_duplicates('division_key')[DIVISION_COLUMNS]
    .reset_index(drop=True)
)
party_counts = audit.groupby('division_key')['party'].nunique().rename('party_count')
divisions = divisions.merge(party_counts, on='division_key', how='left', validate='one_to_one')

def text_hash(value):
    text = '' if pd.isna(value) else str(value).strip().lower()
    return hashlib.sha1(text.encode('utf-8')).hexdigest()[:16]

divisions['motion_text_group_id'] = divisions['motion_text_clean'].map(text_hash)
display(divisions.head(3))
print(f'Division-level 数据：{len(divisions):,} 行')
print(f"唯一 motion text：{divisions['motion_text_group_id'].nunique():,}")

,division_key,motion_date,motion_year,motion_title_clean,motion_text_clean,motion_type,division_name_clean,legislation_name_clean,legislation_id,split,motion_polarity,polarity_rule,polarity_confidence,government_party,main_opposition_party,party_count,motion_text_group_id
0,pw-2016-01-06-157-commons,2016-01-06,2016,Opposition Day — [14th Allotted Day] — Universal Credit Work Allowance,"I beg to move, That this House calls on the Government to reverse its decision to cut the universal credit work allowance, which is due ...",other,Opposition Day — Universal Credit Work Allowance — Decision to Cut,NaN,NaN,train,-1.0,reverse_or_remove,high,conservative,labour,4,de78823e4c252793
1,pw-2016-01-06-158-commons,2016-01-06,2016,Opposition Day — [14th Allotted Day] — Universal Credit Work Allowance: Flooding,"Question put, The House divided: Ayes 216, Noes 281.",other,Opposition Day — [14th Allotted Day] — Universal Credit Work Allowance — Flooding,NaN,NaN,train,NaN,manual_review,unknown,conservative,labour,4,db5810cc6b774a85
2,pw-2016-01-07-159-commons,2016-01-07,2016,State Pension Age (Women),"Resolved, That this House, while welcoming the equalisation of the state pension age, is concerned that the acceleration of that equalis...",other,State Pension Age (Women),NaN,NaN,train,NaN,manual_review,unknown,conservative,labour,4,327ca0803e1a5fff


Division-level 数据：2,091 行
唯一 motion text：1,867


## 4. 解析 Bill Sponsor

Government organisation 是政府背书的强证据。只有执政党议员而没有政府部门 organisation，不足以证明这是 Government Bill。

In [16]:
GOVERNMENT_ORG_PATTERN = re.compile(
    r'department|ministry|hm treasury|home office|cabinet office|government office|'
    r'attorney general|advocate general|office of the secretary|leader of the house',
    flags=re.IGNORECASE,
)

def parse_sponsors(value):
    if pd.isna(value):
        return [], False
    try:
        parsed = ast.literal_eval(str(value))
        return parsed if isinstance(parsed, list) else [], True
    except (ValueError, SyntaxError):
        return [], False

def sponsor_features(value):
    records, parse_ok = parse_sponsors(value)
    parties = set()
    organisations = set()

    # Sponsor 可能只有 member、只有 organisation，或其中一个字段明确为 None。
    # 逐项检查类型，避免在 None 或异常元素上调用 .get()。
    for item in records:
        if not isinstance(item, dict):
            parse_ok = False
            continue

        member = item.get('member')
        if isinstance(member, dict) and member.get('party'):
            parties.add(str(member['party']).strip())

        organisation = item.get('organisation')
        if isinstance(organisation, dict) and organisation.get('name'):
            organisations.add(str(organisation['name']).strip())

    parties = sorted(parties)
    organisations = sorted(organisations)
    government_orgs = [org for org in organisations if GOVERNMENT_ORG_PATTERN.search(org)]
    return pd.Series({
        'sponsor_parse_ok': parse_ok,
        'bill_sponsor_parties': ' | '.join(parties) if parties else pd.NA,
        'bill_sponsor_organisations': ' | '.join(organisations) if organisations else pd.NA,
        'bill_government_organisations': ' | '.join(government_orgs) if government_orgs else pd.NA,
        'bill_has_government_organisation': bool(government_orgs),
    })

bill_features = pd.concat([bill_info.copy(), bill_info['sponsors'].apply(sponsor_features)], axis=1)
display(bill_features[[
    'billId', 'shortTitle', 'bill_sponsor_parties',
    'bill_sponsor_organisations', 'bill_has_government_organisation'
]].head(10))

,billId,shortTitle,bill_sponsor_parties,bill_sponsor_organisations,bill_has_government_organisation
0,1139,Financial Services (Banking Reform) Act,Conservative,HM Treasury,True
1,2172,Trade Bill,Conservative,Department for International Trade,True
2,2801,National Security and Investment Act 2021,Conservative,"Department for Business, Energy and Industrial Strategy",True
3,2461,Environment Bill,Conservative,"Department for Environment, Food and Rural Affairs",True
4,2783,Covert Human Intelligence Sources (Criminal Conduct) Act 2021,Conservative,Home Office,True
5,2730,Fire Safety Act 2021,Conservative,Home Office,True
6,2464,Telecommunications Infrastructure (Leasehold Property) Bill,Conservative,"Department for Culture, Media and Sport",True
7,2838,Contingencies Fund Act 2021,Conservative,HM Treasury,True
8,2839,"Police, Crime, Sentencing and Courts Act 2022",Conservative,Ministry of Justice,True
9,4042,Finance (No. 2) Bill,Labour,HM Treasury,True


## 5. 使用标准化标题关联 Bill

all.csv 的 legislation_id 与 bill_info 的 billId 来自不同 ID 系统，不能直接关联。本步骤对 legislation_name 和 shortTitle 做标准化，只自动接受唯一的精确标准化匹配。

In [17]:
def normalize_bill_title(value):
    if pd.isna(value):
        return pd.NA
    text = html.unescape(str(value))
    text = unicodedata.normalize('NFKD', text).lower()
    text = re.sub(r'\b(?:19|20)\d{2}\b', ' ', text)
    text = re.sub(r'\b(?:act|bill|lords)\b', ' ', text)
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    text = ' '.join(text.split())
    return text if text else pd.NA

bill_features['bill_title_key'] = bill_features['shortTitle'].map(normalize_bill_title)
bill_key_counts = bill_features['bill_title_key'].value_counts(dropna=True)
unique_bill_keys = set(bill_key_counts[bill_key_counts.eq(1)].index)
unique_bill_lookup = (
    bill_features.loc[bill_features['bill_title_key'].isin(unique_bill_keys), [
        'bill_title_key', 'billId', 'shortTitle', 'introducedSessionId', 'lastUpdate',
        'bill_sponsor_parties', 'bill_sponsor_organisations',
        'bill_government_organisations', 'bill_has_government_organisation',
        'sponsor_parse_ok',
    ]]
    .rename(columns={
        'billId': 'matched_bill_id',
        'shortTitle': 'matched_bill_title',
        'introducedSessionId': 'matched_bill_session_id',
        'lastUpdate': 'matched_bill_last_update',
    })
)

divisions['source_bill_title_key'] = divisions['legislation_name_clean'].map(normalize_bill_title)
divisions = divisions.merge(
    unique_bill_lookup,
    left_on='source_bill_title_key',
    right_on='bill_title_key',
    how='left',
    validate='many_to_one',
)
# 同名 Bill 可能跨届重复出现。只匹配标题会把新议案连到多年以前的同名法案。
divisions['matched_bill_last_year'] = pd.to_datetime(
    divisions['matched_bill_last_update'], errors='coerce', format='mixed'
).dt.year.astype('Int64')
divisions['bill_motion_year_gap'] = (
    divisions['motion_year'].astype('Int64') - divisions['matched_bill_last_year']
)
title_match = divisions['matched_bill_id'].notna()
temporal_match = title_match & divisions['bill_motion_year_gap'].abs().le(3)
temporal_rejection = title_match & ~temporal_match

# 保留被拒绝的候选，便于审计；但不能让它进入政府背书推断。
divisions['rejected_candidate_bill_id'] = divisions['matched_bill_id'].where(temporal_rejection)
divisions['rejected_candidate_bill_title'] = divisions['matched_bill_title'].where(temporal_rejection)
divisions['bill_link_rejection_reason'] = np.where(
    temporal_rejection, 'temporal_mismatch', pd.NA
)
bill_match_columns = [
    'matched_bill_id', 'matched_bill_title', 'matched_bill_session_id',
    'matched_bill_last_update', 'bill_title_key',
    'bill_sponsor_parties', 'bill_sponsor_organisations',
    'bill_government_organisations', 'bill_has_government_organisation',
    'sponsor_parse_ok',
]
divisions.loc[temporal_rejection, bill_match_columns] = pd.NA
divisions['bill_link_method'] = np.where(
    temporal_match, 'exact_normalized_title_temporal_check', pd.NA
)
divisions['bill_link_confidence'] = np.select(
    [temporal_match, temporal_rejection],
    ['high', 'rejected'],
    default='unmatched',
)

total_divisions = len(divisions)
named_divisions = int(divisions['legislation_name_clean'].notna().sum())
matched_divisions = int(divisions['matched_bill_id'].notna().sum())
government_sponsor_divisions = int((
    divisions['matched_bill_id'].notna()
    & divisions['bill_has_government_organisation'].eq(True)
).sum())
link_summary = pd.Series({
    '总议案数': total_divisions,
    '有legislation_name的议案数': named_divisions,
    'Bill精确匹配议案数': matched_divisions,
    '因时间不合理而拒绝的同名Bill匹配数': int(temporal_rejection.sum()),
    'Bill精确匹配占全部议案比例_pct': round(matched_divisions / total_divisions * 100, 2),
    'Bill精确匹配占有legislation_name议案比例_pct': round(matched_divisions / named_divisions * 100, 2) if named_divisions else 0,
    '匹配且有政府部门Sponsor的议案数': government_sponsor_divisions,
    '政府部门Sponsor证据占全部议案比例_pct': round(government_sponsor_divisions / total_divisions * 100, 2),
}, name='value')
display(link_summary.to_frame())

,value
总议案数,2091.00
有legislation_name的议案数,351.00
Bill精确匹配议案数,286.00
因时间不合理而拒绝的同名Bill匹配数,35.00
Bill精确匹配占全部议案比例_pct,13.68
Bill精确匹配占有legislation_name议案比例_pct,81.48
匹配且有政府部门Sponsor的议案数,260.00
政府部门Sponsor证据占全部议案比例_pct,12.43


## 6. 为未匹配标题生成候选

模糊匹配只生成前三个候选和相似度，不自动写入 matched_bill_id。Finance Bill 等重复名称必须结合年份与 session 人工确认。

In [18]:
bill_candidate_table = bill_features[[
    'billId', 'shortTitle', 'introducedSessionId', 'lastUpdate', 'bill_title_key'
]].dropna(subset=['bill_title_key']).drop_duplicates()

unmatched_source_titles = (
    divisions.loc[
        divisions['legislation_name_clean'].notna() & divisions['matched_bill_id'].isna(),
        ['legislation_name_clean', 'source_bill_title_key'],
    ]
    .drop_duplicates()
)

candidate_rows = []
for source_row in unmatched_source_titles.itertuples(index=False):
    scored = []
    for bill_row in bill_candidate_table.itertuples(index=False):
        score = SequenceMatcher(None, source_row.source_bill_title_key, bill_row.bill_title_key).ratio()
        scored.append((score, bill_row))
    for rank, (score, bill_row) in enumerate(sorted(scored, key=lambda item: item[0], reverse=True)[:3], start=1):
        candidate_rows.append({
            'source_legislation_name': source_row.legislation_name_clean,
            'candidate_rank': rank,
            'similarity': round(score, 4),
            'candidate_bill_id': bill_row.billId,
            'candidate_bill_title': bill_row.shortTitle,
            'candidate_session_id': bill_row.introducedSessionId,
            'candidate_last_update': bill_row.lastUpdate,
            'human_accept_match': pd.NA,
        })

fuzzy_candidates = pd.DataFrame(candidate_rows)
display(fuzzy_candidates.head(15))

,source_legislation_name,candidate_rank,similarity,candidate_bill_id,candidate_bill_title,candidate_session_id,candidate_last_update,human_accept_match
0,Finance Bill (2024),1,0.7368,4042,Finance (No. 2) Bill,39,2026-03-06T18:05:32.1586889,<NA>
1,Finance Bill (2024),2,0.5600,2792,Financial Services Act 2021,35,2023-04-13T10:44:51.4011663,<NA>
2,Finance Bill (2024),3,0.4615,3701,Football Governance Bill,38,2024-09-23T14:27:18.5810805,<NA>
3,Renters (Reform) Bill,1,0.6429,3764,Renters' Rights Act 2025,39,2026-02-27T16:35:08.969809,<NA>
4,Renters (Reform) Bill,2,0.4681,1139,Financial Services (Banking Reform) Act,25,2013-12-19T10:38:32,<NA>
5,Renters (Reform) Bill,3,0.4651,3523,Leasehold and Freehold Reform Act 2024,38,2025-04-15T12:05:10.3433099,<NA>
6,Criminal Justice Bill,1,0.8421,283,Criminal Justice (Raves) Bill,21,2008-10-30T18:56:00,<NA>
7,Criminal Justice Bill,2,0.5294,2792,Financial Services Act 2021,35,2023-04-13T10:44:51.4011663,<NA>
8,Criminal Justice Bill,3,0.4490,3774,Terminally Ill Adults (End of Life) Bill,39,2026-03-02T09:11:05.7748772,<NA>
9,Employment Rights Bill,1,1.0000,1446,Employment Rights Bill,27,2015-03-30T13:46:48,<NA>


## 7. 生成政府背书建议

此字段是建议，不是 gold label。Bill sponsor 中出现政府部门时可高置信判断基础 Bill 为政府背书。反对党 sponsor 且没有政府部门时给出中置信非政府建议。Opposition Day 因政策对象可能与 motion 本身不同，保留人工判断。

In [19]:
def suggest_government_backing(row):
    # 安全地把缺失值转换为空字符串，避免 pd.NA 参与布尔判断。
    def safe_text(value):
        return '' if pd.isna(value) else str(value)

    text = ' '.join([
        safe_text(row.get('division_name_clean')),
        safe_text(row.get('motion_title_clean')),
        safe_text(row.get('motion_text_clean')),
    ]).lower()
    has_government_org = row.get('bill_has_government_organisation')
    if pd.notna(has_government_org) and bool(has_government_org):
        return 1, 'bill_sponsor_government_organisation', 'high'
    if re.search(r'\bgovernment (?:motion|amendment)\b', text):
        return 1, 'explicit_government_motion', 'high'
    sponsor_parties = safe_text(row.get('bill_sponsor_parties')).lower()
    government_party = safe_text(row.get('government_party')).lower()
    if pd.notna(row.get('matched_bill_id')) and sponsor_parties:
        if government_party and government_party not in sponsor_parties:
            return 0, 'matched_bill_opposition_sponsor_no_government_org', 'medium'
    if 'opposition day' in text:
        return pd.NA, 'opposition_day_requires_object_alignment', 'unknown'
    return pd.NA, 'manual_review', 'unknown'

backing_suggestions = divisions.apply(suggest_government_backing, axis=1, result_type='expand')
backing_suggestions.columns = [
    'suggested_object_government_backed',
    'suggested_government_backing_rule',
    'suggested_government_backing_confidence',
]
divisions = pd.concat([divisions, backing_suggestions], axis=1)
divisions['suggested_object_government_backed'] = (
    divisions['suggested_object_government_backed'].astype('Int64')
)
display(divisions.groupby([
    'suggested_government_backing_confidence', 'suggested_object_government_backed'
], dropna=False).size().to_frame('divisions'))

,,divisions
suggested_government_backing_confidence,suggested_object_government_backed,
high,1,283
unknown,<NA>,1808


## 8. 标记程序性议案与人工复核优先级

程序性议案保留在审计文件中，但默认不进入政策立场训练，也不进入必做 polarity 标注队列。

In [20]:
procedural_text = (
    divisions['motion_title_clean'].fillna('') + ' ' + divisions['motion_text_clean'].fillna('')
).str.lower()
divisions['procedural_exclusion_reason'] = np.select(
    [
        divisions['motion_type'].isin(PROCEDURAL_EXCLUDE_TYPES),
        procedural_text.str.contains(
            r'\breasons committee\b|\bcommittee be appointed to draw up a reason\b',
            regex=True,
        ),
    ],
    ['procedural_motion_type', 'reasons_committee_text'],
    default=pd.NA,
)
divisions['eligible_for_stance_model'] = divisions['procedural_exclusion_reason'].isna()
divisions['procedural_exclusion_reason'] = np.where(
    divisions['eligible_for_stance_model'], pd.NA,
    'excluded_motion_type:' + divisions['motion_type'].astype(str),
)

def review_reason(row):
    if not row['eligible_for_stance_model']:
        return 'procedural_excluded'
    if pd.isna(row['motion_polarity']):
        return 'unknown_polarity_required'
    if row['polarity_confidence'] == 'medium':
        return 'medium_polarity_recommended'
    return 'no_polarity_review_required'

divisions['polarity_review_reason'] = divisions.apply(review_reason, axis=1)
priority_conditions = [
    divisions['polarity_review_reason'].eq('unknown_polarity_required') & divisions['split'].eq('test'),
    divisions['polarity_review_reason'].eq('unknown_polarity_required') & divisions['split'].eq('validation'),
    divisions['polarity_review_reason'].eq('unknown_polarity_required') & divisions['split'].eq('train'),
    divisions['polarity_review_reason'].eq('medium_polarity_recommended'),
]
divisions['review_priority'] = np.select(
    priority_conditions, ['P0', 'P1', 'P2', 'P3'], default='not_required'
)

display(pd.crosstab(divisions['polarity_review_reason'], divisions['split']))
display(pd.crosstab(divisions['review_priority'], divisions['motion_type']))

split,test,train,validation
polarity_review_reason,,,
medium_polarity_recommended,56,137,38
no_polarity_review_required,256,773,100
procedural_excluded,11,54,8
unknown_polarity_required,115,489,54


motion_type,add_clause_to_bill,adjournment,amendment,approve_statutory_instrument,bill_introduction,closure,committee_clause,eu_document_scrutiny,financial,government_agenda,humble_address,lords_amendment,other,private_sitting,programme,proposed_clause,reasoned_amendment,reasons_committee,revoke_statutory_instrument,second_stage,standing_order_change,ten_minute_rule,third_stage,timetable_change
review_priority,,,,,,,,,,,,,,,,,,,,,,,,
P0,0,0,54,0,0,0,0,0,0,0,0,22,24,0,0,1,13,0,0,0,0,1,0,0
P1,0,0,25,0,0,0,0,0,2,0,1,3,14,0,0,0,8,0,0,0,0,0,1,0
P2,0,0,212,0,0,0,0,8,0,6,9,41,195,0,0,2,11,0,0,1,0,3,1,0
P3,0,0,0,0,18,0,0,0,0,0,0,0,0,0,0,206,2,0,2,2,0,0,1,0
not_required,85,6,173,217,3,6,24,0,0,20,1,402,32,25,13,12,7,8,4,77,2,4,75,6


## 9. 生成人工标注队列

同一 motion_text_group_id 的文本可能重复。标注时可以先处理一条，再检查能否安全传播到同组议案。不要在没有核对政策对象的情况下自动传播。

In [21]:
REVIEW_EXPORT_COLUMNS = [
    'review_priority', 'polarity_review_reason', 'split',
    'division_key', 'motion_date', 'motion_type',
    'motion_title_clean', 'motion_text_clean', 'motion_text_group_id',
    'legislation_name_clean', 'matched_bill_id', 'matched_bill_title',
    'bill_link_method', 'bill_link_rejection_reason',
    'rejected_candidate_bill_id', 'rejected_candidate_bill_title',
    'bill_sponsor_parties', 'bill_sponsor_organisations',
    'bill_government_organisations', 'government_party',
    'motion_polarity', 'polarity_rule', 'polarity_confidence',
    'suggested_object_government_backed',
    'suggested_government_backing_rule',
    'suggested_government_backing_confidence',
]

polarity_review_queue = (
    divisions.loc[
        divisions['polarity_review_reason'].isin([
            'unknown_polarity_required', 'medium_polarity_recommended'
        ]),
        REVIEW_EXPORT_COLUMNS,
    ]
    .copy()
    .sort_values(['review_priority', 'motion_text_group_id', 'motion_date', 'division_key'])
    .reset_index(drop=True)
)

polarity_review_queue['policy_object'] = pd.NA
polarity_review_queue['aye_action'] = pd.NA
polarity_review_queue['human_motion_polarity'] = pd.NA
polarity_review_queue['human_object_government_backed'] = pd.NA
polarity_review_queue['evidence_span'] = pd.NA
polarity_review_queue['reviewer'] = pd.NA
polarity_review_queue['review_status'] = 'pending'
polarity_review_queue['review_notes'] = pd.NA

print(f'人工复核队列：{len(polarity_review_queue):,} 个唯一 division')
print(f"其中 unknown polarity：{polarity_review_queue['motion_polarity'].isna().sum():,}")
print(f"唯一文本组：{polarity_review_queue['motion_text_group_id'].nunique():,}")
display(polarity_review_queue.head(10))

人工复核队列：889 个唯一 division
其中 unknown polarity：658
唯一文本组：878


,review_priority,polarity_review_reason,split,division_key,motion_date,motion_type,motion_title_clean,motion_text_clean,motion_text_group_id,legislation_name_clean,matched_bill_id,matched_bill_title,bill_link_method,bill_link_rejection_reason,rejected_candidate_bill_id,rejected_candidate_bill_title,bill_sponsor_parties,bill_sponsor_organisations,bill_government_organisations,government_party,motion_polarity,polarity_rule,polarity_confidence,suggested_object_government_backed,suggested_government_backing_rule,suggested_government_backing_confidence,policy_object,aye_action,human_motion_polarity,human_object_government_backed,evidence_span,reviewer,review_status,review_notes
0,P0,unknown_polarity_required,test,pw-2025-03-12-122-commons,2025-03-12,amendment,Employment Rights Bill: Schedule 5 - Legislation subject to enforcement under part 5,"Amendments made: 246, page 155, line 2, at end insert_ __ocial Security Contributions and Benefits (Northern Ireland) Act 1992 7A Sectio...",03ccee52f0c8dce7,Employment Rights Bill,NaN,<NA>,<NA>,temporal_mismatch,1446.0,Employment Rights Bill,<NA>,<NA>,<NA>,labour,NaN,manual_review,unknown,<NA>,manual_review,unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>
1,P0,unknown_polarity_required,test,pw-2025-05-14-198-commons,2025-05-14,lords_amendment,Clause 28 - DVS trust framework,"Resolved, That this House insists on Commons Amendment 32 to which the Lords have disagreed and disagrees with the Lords in their Amendm...",05d8cdd163a7e816,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,labour,NaN,multi_object_manual_review,unknown,<NA>,manual_review,unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>
2,P0,unknown_polarity_required,test,pw-2025-07-09-264-commons,2025-07-09,amendment,Amendment: Universal Credit and Personal Independence Payment Bill: Schedule 1 - Amendments to the Universal Credit Regulations 2013 in ...,"Amendment proposed: 38, page 6, leave out lines 33 and 34 and insert_ _(b) has either_ (i) been entitled to an award of universal credit...",06f2e1e5426f5df6,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,labour,NaN,multi_object_manual_review,unknown,<NA>,manual_review,unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>
3,P0,unknown_polarity_required,test,pw-2025-11-17-357-commons,2025-11-17,amendment,Clause 12 - Procedure for regulations under section 11,"Amendment proposed: 5, in clause 12, page 9, line 2, at end insert_ _(aa) relating to the charging of fees under section 11(3)(c),__(And...",085662f7801ca5ff,Biodiversity Beyond National Jurisdiction Bill (2026),4016.0,Biodiversity Beyond National Jurisdiction Act 2026,exact_normalized_title_temporal_check,<NA>,NaN,NaN,Labour,"Foreign, Commonwealth and Development Office",<NA>,labour,NaN,manual_review,unknown,<NA>,manual_review,unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>
4,P0,unknown_polarity_required,test,pw-2025-06-18-237-commons,2025-06-18,amendment,New Clause 121: Crime and Policing Bill,New Clause 121 - Amendment of Possession of extreme pornographic images _(1) Section 63 of the Criminal Justice and Immigration Act 2008...,09cfec75a4bc1e8f,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,labour,NaN,manual_review,unknown,<NA>,manual_review,unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>
5,P0,unknown_polarity_required,test,pw-2025-12-03-380-commons,2025-12-03,amendment,Clause 40 - Certain schemes providing money purchase benefits: scale and asset allocation,"Amendment proposed: 15, page 46, line 9, leave out from __efore_ to the end of the subsection and insert __mplementing the first set of ...",0adf5c111d583b70,Pension Schemes Bill,3982.0,Pension Schemes Bill,exact_normalized_title_temporal_check,<NA>,NaN,NaN,Labour,Department for Work and Pensions,Department for Work and Pensions,labour,NaN,multi_object_manual_review,unknown,1,bill_sponsor_government_organisation,high,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>
6,P0,unknown_polarity_required,test,pw-2026-04-15-488-commons,2026-04-15,lords_amendment,Clause 56 - Functions of adjudicator

## 10. 生成 High-confidence 质量抽样

High confidence 也不能被默认视为完全正确。固定随机抽样用于估计规则误差，并决定是否需要修改规则。

In [22]:
high_confidence_pool = divisions.loc[
    divisions['eligible_for_stance_model']
    & divisions['polarity_confidence'].eq('high')
    & divisions['motion_polarity'].notna()
].copy()
sample_size = min(HIGH_CONFIDENCE_QC_SIZE, len(high_confidence_pool))
high_confidence_qc = high_confidence_pool.sample(
    n=sample_size, random_state=RANDOM_SEED
)[REVIEW_EXPORT_COLUMNS].copy()
high_confidence_qc['policy_object'] = pd.NA
high_confidence_qc['aye_action'] = pd.NA
high_confidence_qc['human_motion_polarity'] = pd.NA
high_confidence_qc['is_rule_correct'] = pd.NA
high_confidence_qc['evidence_span'] = pd.NA
high_confidence_qc['reviewer'] = pd.NA
high_confidence_qc['review_status'] = 'pending'
high_confidence_qc['review_notes'] = pd.NA
display(high_confidence_qc.head())

,review_priority,polarity_review_reason,split,division_key,motion_date,motion_type,motion_title_clean,motion_text_clean,motion_text_group_id,legislation_name_clean,matched_bill_id,matched_bill_title,bill_link_method,bill_link_rejection_reason,rejected_candidate_bill_id,rejected_candidate_bill_title,bill_sponsor_parties,bill_sponsor_organisations,bill_government_organisations,government_party,motion_polarity,polarity_rule,polarity_confidence,suggested_object_government_backed,suggested_government_backing_rule,suggested_government_backing_confidence,policy_object,aye_action,human_motion_polarity,is_rule_correct,evidence_span,reviewer,review_status,review_notes
502,not_required,no_polarity_review_required,train,pw-2019-10-23-7-commons,2019-10-23,approve_statutory_instrument,Deferred Divisions: EXITING THE EUROPEAN UNION (AUDITORS),"That the draft Statutory Auditors, Third Country Auditors and International Accounting Standards (Amendment) (EU Exit) Regulations 2019,...",4ffe8520cab5ca7d,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,conservative,1.0,approve,high,<NA>,manual_review,unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>
265,not_required,no_polarity_review_required,train,pw-2018-06-12-167-commons,2018-06-12,lords_amendment,Disagree: Lords amendment 128,"Motion made, and Question put, That this House disagrees with Lords amendment 128._(Mr David Davis.)",ddc7927960b680db,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,conservative,-1.0,disagree_lords,high,<NA>,manual_review,unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>
1820,not_required,no_polarity_review_required,test,pw-2025-06-20-241-commons,2025-06-20,amendment,Clause 3 - Capacity,"Amendment proposed: 24, in clause 3, page 2, line 18, at end insert __xcept that section 1(2) of that Act shall not apply_._(Daniel Fran...",16f1fd2d123b8881,Terminally Ill Adults (End of Life) Bill,3774.0,Terminally Ill Adults (End of Life) Bill,exact_normalized_title_temporal_check,<NA>,NaN,NaN,Labour,<NA>,<NA>,labour,1.0,insert_or_add,high,<NA>,manual_review,unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>
1523,not_required,no_polarity_review_required,validation,pw-2024-03-18-106-commons,2024-03-18,lords_amendment,Disagree: Lords amendment 9,"Motion made, and Question put, That this House disagrees with Lords amendment 9._(Michael Tomlinson.)",5faab22b3d3e4a0f,Safety of Rwanda (Asylum and Immigration) Bill (2024),3540.0,Safety of Rwanda (Asylum and Immigration) Act 2024,exact_normalized_title_temporal_check,<NA>,NaN,NaN,Conservative,Home Office,Home Office,conservative,-1.0,disagree_lords,high,1,bill_sponsor_government_organisation,high,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>
1423,not_required,no_polarity_review_required,train,pw-2023-11-15-4-commons,2023-11-15,government_agenda,"Amendment: Violence Reduction, Policing and Criminal Justice","Amendment proposed: (h), at the end of the Question to add: __ut respectfully regret that the Gracious Speech fails to include measures ...",dc9ba67d43dd3cfc,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,conservative,1.0,insert_or_add,high,<NA>,manual_review,unknown,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>


## 11. 导出文件

division_annotation_base 是完整的 division-level 审计表。polarity_review_queue 是需要人工处理的主文件。fuzzy_bill_candidates 只能作为候选参考。

In [23]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bill_linkage_audit = divisions[[
    'division_key', 'motion_date', 'legislation_name_clean',
    'source_bill_title_key', 'matched_bill_id', 'matched_bill_title',
    'bill_link_method', 'bill_link_confidence', 'bill_motion_year_gap',
    'bill_link_rejection_reason', 'rejected_candidate_bill_id',
    'rejected_candidate_bill_title',
    'bill_sponsor_parties', 'bill_sponsor_organisations',
    'bill_government_organisations', 'bill_has_government_organisation',
]].copy()

summary = {
    'division_count': int(len(divisions)),
    'eligible_divisions': int(divisions['eligible_for_stance_model'].sum()),
    'procedural_excluded_divisions': int((~divisions['eligible_for_stance_model']).sum()),
    'exact_bill_matches': int(divisions['matched_bill_id'].notna().sum()),
    'rejected_temporal_bill_matches': int((
        divisions['bill_link_rejection_reason'] == 'temporal_mismatch'
    ).sum()),
    'bill_matches_with_government_organisation': int((
        divisions['matched_bill_id'].notna()
        & divisions['bill_has_government_organisation'].fillna(False)
    ).sum()),
    'unknown_polarity_divisions_before_procedural_filter': int(divisions['motion_polarity'].isna().sum()),
    'required_unknown_polarity_reviews': int((
        divisions['polarity_review_reason'] == 'unknown_polarity_required'
    ).sum()),
    'recommended_medium_polarity_reviews': int((
        divisions['polarity_review_reason'] == 'medium_polarity_recommended'
    ).sum()),
    'review_queue_rows': int(len(polarity_review_queue)),
    'review_queue_unique_text_groups': int(polarity_review_queue['motion_text_group_id'].nunique()),
    'high_confidence_qc_rows': int(len(high_confidence_qc)),
}

divisions.to_csv(OUTPUT_DIR / 'division_annotation_base.csv', index=False)
bill_linkage_audit.to_csv(OUTPUT_DIR / 'bill_linkage_audit.csv', index=False)
fuzzy_candidates.to_csv(OUTPUT_DIR / 'fuzzy_bill_candidates.csv', index=False)
polarity_review_queue.to_csv(OUTPUT_DIR / 'polarity_review_queue.csv', index=False)
high_confidence_qc.to_csv(OUTPUT_DIR / 'high_confidence_qc_sample.csv', index=False)
with (OUTPUT_DIR / 'review_summary.json').open('w', encoding='utf-8') as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

print('已生成：')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {path.name}')
display(pd.Series(summary, name='value').to_frame())

已生成：
  bill_linkage_audit.csv
  division_annotation_base.csv
  fuzzy_bill_candidates.csv
  high_confidence_qc_manually_reviewed.csv
  high_confidence_qc_sample.csv
  manual_validation_report.md
  polarity_review_queue.csv
  review_summary.json


/var/folders/tk/r_n7kv150lj5lky4nj__dc7r0000gn/T/ipykernel_34753/2004473609.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & divisions['bill_has_government_organisation'].fillna(False)


,value
division_count,2091
eligible_divisions,2018
procedural_excluded_divisions,73
exact_bill_matches,286
rejected_temporal_bill_matches,35
bill_matches_with_government_organisation,260
unknown_polarity_divisions_before_procedural_filter,709
required_unknown_polarity_reviews,658
recommended_medium_polarity_reviews,231
review_queue_rows,889


## 12. 人工标注规则

每个 division 依次回答：

1. policy_object：最终希望预测政党对哪个 Bill、Clause、Amendment 或政策的立场；
2. aye_action：投 Aye 会采纳或保留该对象，还是删除、撤回或否决该对象；
3. human_motion_polarity：采纳或保留填 1，删除、撤回或否决填 -1，不适合政策立场建模填 exclude；
4. human_object_government_backed：针对同一个 policy_object 填 1、0 或 unknown；
5. evidence_span：复制支持判断的最短原文；
6. review_status：完成后改为 approved。

不要因为 Bill 由政府 sponsor，就自动把反对党 amendment 标成政府背书。先确认 policy_object 是基础 Bill/Clause 还是 amendment。

## 13. 最终检查

In [24]:
assert divisions['division_key'].is_unique, 'division_annotation_base 中 division_key 不唯一。'
assert polarity_review_queue['division_key'].is_unique, '人工复核队列中 division_key 不唯一。'
assert 'reform' not in set(audit['party'].dropna()), '数据中仍包含 Reform。'
assert set(polarity_review_queue['review_status']) == {'pending'}, '新队列应全部处于 pending 状态。'
assert not polarity_review_queue['polarity_review_reason'].eq('procedural_excluded').any(), '程序性议案进入了人工 polarity 队列。'

print('第二步数据准备完成。')
print('下一步：先完成人工复核队列和 High-confidence 抽样，再合并 gold labels。')

第二步数据准备完成。
下一步：先完成人工复核队列和 High-confidence 抽样，再合并 gold labels。
